# 08_逐帧CNN与时间平均

安排：第08课，接在07课之后、09课之前。目标：跟踪从视频到逐帧分类再到时间平均的shape，理解平均会丢弃顺序。

来源：Word手册的CNN/表示学习概念；飞书《A 观行知意 学习指导》关于逐帧CNN与时间聚合的介绍。飞书指出其Kaggle示例使用热成像；这里读取已有深度/IR，只借鉴思路，不导入热成像数据或权重。完整对照见README.md。

本示例没有训练，卷积和分类头为随机参数。可以运行并观察形状，不能作为比赛模型或效果证据。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/learning.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT / 'tools'))
import learning as L
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


## 空间特征与片段分类

同一个2D CNN处理每帧；分类头每帧返回40个logits。本例在类别维softmax，然后沿时间平均。它是明确选择的一种教学实现，未声称与Kaggle原代码逐行一致。


In [ ]:
import torch
from torch import nn
torch.set_num_threads(4)
torch.manual_seed(42)

class FrameMeanClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 8, kernel_size=3, padding=1),
            nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Linear(8, 40)

    def forward(self, video):
        # video：[B,C,T,H,W]。先把时间移到通道前，再合并B与T。
        b, c, t, h, w = video.shape
        frames = video.permute(0, 2, 1, 3, 4).reshape(b*t, c, h, w)
        features = self.encoder(frames)         # [B*T,8]
        logits = self.head(features).reshape(b, t, 40)
        probabilities = logits.softmax(dim=-1) # 每帧40类分数
        return probabilities.mean(dim=1), logits

model = FrameMeanClassifier().eval()


## 真实片段经过教学网络

4通道来自伪彩深度3通道加IR1通道。permutation交换轴与reshape合并维度分开做；不能用reshape直接代替轴交换。


In [ ]:
clip, true_label, classes = L.load_clip('validation', 0, frames=16, size=128)
video = torch.from_numpy(clip).unsqueeze(0)
with torch.inference_mode():
    clip_probs, frame_logits = model(video)
assert video.shape == (1, 4, 16, 128, 128)
assert frame_logits.shape == (1, 16, 40)
assert clip_probs.shape == (1, 40)
torch.testing.assert_close(clip_probs.sum(dim=-1), torch.ones(1))
print('输入BCTHW：', tuple(video.shape))
print('每帧分数：', tuple(frame_logits.shape))
print('聚合结果：', tuple(clip_probs.shape))
print('只检查形状；随机教学模型尚未学会识别动作，不报告其准确率。')


## 时间平均的局限

把同一组帧倒序，并没有改变每帧图像及其分类分数的集合。没有顺序编码的平均值不能知道哪一帧先发生。


In [ ]:
with torch.inference_mode():
    reverse_probs, _ = model(video.flip(dims=[2]))
torch.testing.assert_close(clip_probs, reverse_probs, atol=1e-6, rtol=1e-5)
print('同一组帧倒序后，逐帧CNN＋时间平均仍然一致。')
print('这个模型没有位置编码、递归状态或时间卷积；不能从简单平均恢复帧顺序。')


## 核对“平均”的对象

先对logits求平均再softmax，与先softmax再平均通常不同。真实移植前要阅读源码，明确在特征、logits还是概率层聚合。


In [ ]:
# 两帧、两类的极小例子，便于手算；与本题40类同一数学关系。
z = torch.tensor([[4., 0.], [0., 1.]])
average_probabilities = z.softmax(-1).mean(0)
probabilities_of_average_logits = z.mean(0).softmax(-1)
print('先softmax再平均：', average_probabilities.tolist())
print('先平均再softmax：', probabilities_of_average_logits.tolist())
assert not torch.allclose(average_probabilities, probabilities_of_average_logits)
print('移植参考代码时必须核对它到底平均了什么，不把两种实现当作相同。')


## 练习

不使用video.flip，写出时间轴倒序索引并送入同一模型，把聚合分数填入your_answer。结果应与原clip_probs相近。再口头解释为何不能据此认为R(2+1)D或LSTM也对帧顺序不敏感。


In [ ]:
your_answer = None  # TODO：创建倒序索引，并调用model
if your_answer is None:
    print('待完成：BCTHW中的时间轴是第2轴；先选同一组帧，再检查分数。')
else:
    torch.testing.assert_close(your_answer, clip_probs, atol=1e-6, rtol=1e-5)
    assert your_answer.shape == (1, 40)
    print('练习通过')


## 过关检查

能逐个解释[B,4,T,H,W]→[B*T,4,H,W]→[B,T,40]→[B,40]；能说明平均与softmax的次序；知道4通道不是4种模态；知道随机教学分数不代表动作识别效果。

现有R(2+1)D34使用空间与时间卷积，本次不替换它，也不改变A/B学习率实验。LSTM和Transformer可在理解这条数据流后选读。

[参考答案](../参考答案/08_逐帧CNN与时间平均.ipynb) · [两份新资料的阅读安排](../README.md) · [飞书指导](https://pcnuxhluoos7.feishu.cn/wiki/JADBwOsvgiWSGVkgt5gcPt39nqh)
